In [2]:
!pip install nibabel

In [17]:
import nibabel as nib
import matplotlib.pyplot as plt

In [18]:
import numpy as np
from ipywidgets import interact, IntSlider
from IPython.display import display

# Load the NIfTI files
ct_img = nib.load('111221_CT.nii.gz')
tumor_img = nib.load('111221_tumor.nii.gz')

ct_data = ct_img.get_fdata()
tumor_data = tumor_img.get_fdata()

# Ensure both images have the same dimensions for overlay
if ct_data.shape != tumor_data.shape:
    print("Warning: CT and tumor image shapes do not match. Resampling might be needed.")
    # For this example, we'll assume they match for a direct overlay.

# Get the number of slices
num_slices = ct_data.shape[2]

def view_slice(slice_idx):
    fig, ax = plt.subplots(figsize=(8, 8))

    # Display CT image (background)
    ax.imshow(ct_data[:, :, slice_idx], cmap='gray')

    # Overlay tumor segmentation. Assuming tumor_data is a binary mask (0 or 1).
    # Create a masked array so that only non-zero values of the tumor are shown.
    tumor_mask_slice = tumor_data[:, :, slice_idx]
    masked_tumor = np.ma.masked_where(tumor_mask_slice == 0, tumor_mask_slice)

    # Define a colormap for the tumor (e.g., red, with transparency).
    # We create a colormap that is transparent for 0 values and red for 1 values.
    cmap = plt.cm.get_cmap('Reds', 2)
    cmap.set_bad(alpha=0) # Make masked (0) values transparent

    # Display the masked tumor over the CT image.
    # 'alpha' controls the transparency of the tumor overlay.
    # 'vmin' and 'vmax' ensure the colormap is correctly scaled for the binary mask.
    ax.imshow(masked_tumor, cmap=cmap, alpha=0.5, vmin=0, vmax=1)

    ax.set_title(f"CT and Tumor Overlay (Slice {slice_idx}/{num_slices-1})")
    ax.axis('off')
    plt.show()

# Create a slider widget for interactive slicing
slice_slider = IntSlider(
    value=num_slices // 2, # Start at the middle slice
    min=0,
    max=num_slices - 1,
    step=1,
    description='Slice:',
    continuous_update=False # Update only when slider is released
)

# Use interact to link the slider to the view_slice function
print("Use the slider below to scroll through the CT and tumor overlay:")
interact(view_slice, slice_idx=slice_slider)

Use the slider below to scroll through the CT and tumor overlay:


interactive(children=(IntSlider(value=98, continuous_update=False, description='Slice:', max=195), Output()), …

<function __main__.view_slice(slice_idx)>

In [19]:
import scipy.ndimage as ndi

# Convert to a binary mask (if not already strictly binary) and label connected components
# A threshold of 0.5 is common for floating point masks, assuming values are 0 or 1.
# If it's strictly 0 or 1 integers, `tumor_data.astype(bool)` would also work.
# Using a structure for 3D connectivity (e.g., 3x3x3 cube) ensures diagonal connections are considered.
labeled_array, num_features = ndi.label(tumor_data > 0.5, structure=np.ones((3,3,3)))

print(f"Number of distinct tumors found: {num_features}")

Number of distinct tumors found: 3


In [20]:
import nibabel as nib

print("--- CT Image Sampling ---")
ct_header = ct_img.header
ct_pixdim = ct_header['pixdim'][1:4] # Get voxel dimensions for x, y, z
print(f"CT Voxel Dimensions (X, Y, Z): {ct_pixdim}")

print("\n--- Tumor Segmentation Image Sampling ---")
tumor_header = tumor_img.header
tumor_pixdim = tumor_header['pixdim'][1:4] # Get voxel dimensions for x, y, z
print(f"Tumor Voxel Dimensions (X, Y, Z): {tumor_pixdim}")

--- CT Image Sampling ---
CT Voxel Dimensions (X, Y, Z): [0.6445312 0.6445312 1.8      ]

--- Tumor Segmentation Image Sampling ---
Tumor Voxel Dimensions (X, Y, Z): [0.6445312 0.6445312 1.8      ]
